<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 17 · Containers, Local Clouds, and Deployment Patterns

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook turns the deployment patterns from Chapter 17 into concrete service specifications.

- Define a handful of container-like services using `ServiceSpec`.
- Build a small compose-style configuration and inspect its YAML.
- Preview a human-readable schedule summary for all services.


In [ ]:
import sys
from pathlib import Path
import subprocess
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/yhilpisch/paatcode.git"
    ROOT_DIR = Path("/content/paatcode")
    if not ROOT_DIR.exists():
        subprocess.run([
            "git",
            "clone",
            REPO_URL,
            str(ROOT_DIR),
        ], check=True)
    PROJECT_ROOT = ROOT_DIR
else:
    PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch17_docker_cloud_production as ch17  # deployment helpers


### 1. Define Services

We mirror the example from the script and declare three small services: a data collector, a strategy loop, and a report runner.

In [ ]:
services = [
    ch17.ServiceSpec(
        name="data_collector",
        image="python:3.12-slim",
        command="python code/ch05_eod_engineering.py",
        schedule="daily 22:15",
        env={"PROFILE": "prod"},
    ),
    ch17.ServiceSpec(
        name="strategy_loop",
        image="python:3.12-slim",
        command="python code/ch07_baseline_strategies.py",
        schedule="market-hours",
        env={"PROFILE": "prod"},
    ),
    ch17.ServiceSpec(
        name="report_runner",
        image="python:3.12-slim",
        command="python code/ch16_reporting_monitoring.py",
        schedule="daily 22:30",
        env={"PROFILE": "prod"},
    ),
]
services


### 2. Compose Configuration and YAML

The helper `build_compose_config` converts these specs into a minimal compose-style dictionary which we then serialise to YAML text.

In [ ]:
config = ch17.build_compose_config(services)
yaml_path = PROJECT_ROOT / "deploy" / "docker-compose.example.yml"
yaml_written = ch17.write_compose_yaml(config, yaml_path)
yaml_written


### 3. Schedule Preview

Finally, we generate a JSON schedule summary that makes it easy to sanity-check when each service is meant to run.

In [ ]:
print(ch17.preview_schedule(services))


### Takeaways

- Even simple data classes and string-based YAML generation make deployment plans explicit and reviewable.
- Keeping configuration in version-controlled text files helps align research environments with live-like setups.
- The same pattern generalises to more complex stacks; only the service list and environment variables change.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">